In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib import util
from kfirlib import llm, prompts, pdfs
from importlib import reload

In [ ]:
reload(util)
reload(pdfs)

In [ ]:
#PDF_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\sophie\Leah Borovoi\86 pdfs\files'
#DATA_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables'
PDF_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Natalie\pdfs\all'
DATA_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Natalie\tables'

In [ ]:
pdf_paths = util.list_files(PDF_DIR, '.pdf')
print(len(pdf_paths))
print(pdf_paths[0])

In [ ]:
#PDF_rest_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\HPV\sophie\Leah Borovoi\1556_pdfs\files'
PDF_rest_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Natalie\pdfs\all'

In [ ]:
pdf_rest_paths = util.list_files(PDF_rest_DIR, '.pdf')
print(len(pdf_rest_paths))
print(pdf_rest_paths[0])

In [ ]:
#pdf_paths = pdf_rest_paths

In [ ]:
eval_process = pdfs.EvalProcess(pdf_rest_paths, model=llm.GEMINI_3_FLASH, criteria=prompts.NATALIE_CRITERIA)
part_gemini = eval_process.evaluate()
part_gemini.head()

In [ ]:
process = pdfs.EvalProcess(pdf_rest_paths, model=llm.GPT_MINI, criteria=prompts.NATALIE_CRITERIA)
part_gpt = process.evaluate()
part_gpt.head()

In [ ]:
part_gpt = process.evaluate()
part_gpt.head()

In [ ]:
part_gemini.rename(columns={'reasoning': 'gemini_reasoning', 'decision': 'gemini_decision'}, inplace=True)
part_gpt.rename(columns={'reasoning': 'gpt_reasoning', 'decision': 'gpt_decision'}, inplace=True)

In [ ]:
set(part_gemini.filename) == set(part_gpt.filename)

In [ ]:
part_gemini.filename.isna().sum()

In [ ]:
part_gpt[['filename', 'gpt_reasoning', 'gpt_decision']].filename.isna().sum()

In [ ]:
merged = pd.merge(part_gemini, part_gpt, on='filename')
merged.head()

In [ ]:
merged.to_excel(os.path.join(DATA_DIR, 'all_pdf_decisions.xlsx'), index=False)

In [ ]:
simple_results = pd.read_excel(os.path.join(DATA_DIR, 'compare_5.xlsx'))
list(simple_results.columns)

In [ ]:
merged = pd.merge(part_gemini, part_gpt, on='filename')
list(merged.columns)

In [ ]:
for base in ['title', 'authors', 'doi', 'pmid']:
    merged[base] = merged[base + '_x'].fillna(merged[base + '_y'])
    print(base, merged[base].isna().sum())

In [ ]:
merged.drop(columns=[col for col in merged.columns if col.endswith('_x') or col.endswith('_y')], inplace=True)
merged = merged[merged.title.notna()].copy()

In [ ]:
merged['title_extract'] = merged.title.apply(extract_letters)
simple_results['title_extract'] = simple_results.title.apply(extract_letters)

In [ ]:
list(merged.columns)

In [ ]:
merged.drop(columns=['doi', 'pmid'], inplace=True)

In [ ]:
merged.rename(columns={'gemini_reasoning': 'reasoning_pdf_gemini',
                       'gemini_decision': 'decision_pdf_gemini',
                       'gpt_reasoning': 'reasoning_pdf_gpt',
                       'gpt_decision': 'decision_pdf_gpt',}, inplace=True)
merged_2 = pd.merge(simple_results, merged, on='title_extract', how='outer', suffixes=('', '_pdf'))
merged_2.head()

In [ ]:
merged_2['title'] = merged_2['title'].fillna(merged_2['title_pdf'])
merged_2['authors'] = merged_2['authors'].fillna(merged_2['authors_pdf'])
merged_2.head()

In [ ]:
len(merged_2), merged_2.filename.isna().sum(), merged_2.year.isna().sum(), merged_2.title.isna().sum()

In [ ]:
merged_2.to_excel(os.path.join(DATA_DIR, 'all_decisions_pdf_and_other.xlsx'), index=False)